In [ ]:

import os, glob, cv2, time, xml.etree.ElementTree as ET
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as T
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {device}")
if device == "cuda":
    print(f"GPU  : {torch.cuda.get_device_name(0)}")
    print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# ============================================================
# PATHS - Pascal VOC 2012
# ============================================================
VOC_ROOT = "/kaggle/input/pascal-voc-2012/VOC2012"

# Auto-find if path differs
if not os.path.exists(VOC_ROOT):
    candidates = glob.glob("/kaggle/input/**/VOC2012", recursive=True)
    if candidates:
        VOC_ROOT = candidates[0]
    else:
        candidates = glob.glob("/kaggle/input/**/JPEGImages", recursive=True)
        if candidates:
            VOC_ROOT = os.path.dirname(candidates[0])
        else:
            raise FileNotFoundError(
                "\n Pascal VOC 2012 not found!\n"
                "Add the dataset: right panel -> Add Data\n"
                "Search: pascal voc 2012 huanghanchina -> Add"
            )

IMG_DIR = os.path.join(VOC_ROOT, "JPEGImages")
ANN_DIR = os.path.join(VOC_ROOT, "Annotations")

n_imgs = len(glob.glob(os.path.join(IMG_DIR, "*.jpg")))
n_anns = len(glob.glob(os.path.join(ANN_DIR, "*.xml")))
print(f"\nImages     : {IMG_DIR}  ({n_imgs} files)")
print(f"Annotations: {ANN_DIR}  ({n_anns} files)")

# ============================================================
# STEP 1 - PARSE VOC XML ANNOTATIONS -> bbox lookup
# ============================================================
print("\nParsing annotations...")
bbox_lookup = defaultdict(list)

for xml_path in glob.glob(os.path.join(ANN_DIR, "*.xml")):
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
        fname = root.find("filename").text
        size  = root.find("size")
        W     = int(size.find("width").text)
        H     = int(size.find("height").text)
        for obj in root.findall("object"):
            bnd = obj.find("bndbox")
            x1  = float(bnd.find("xmin").text) / W
            y1  = float(bnd.find("ymin").text) / H
            x2  = float(bnd.find("xmax").text) / W
            y2  = float(bnd.find("ymax").text) / H
            cx  = (x1 + x2) / 2
            cy  = (y1 + y2) / 2
            bw  = x2 - x1
            bh  = y2 - y1
            bbox_lookup[fname].append((cx, cy, bw, bh))
    except Exception:
        pass

print(f"Parsed {len(bbox_lookup)} images with bounding boxes")

# ============================================================
# STAGE 1 - GrabCut Partitioning
# ============================================================
def grabcut_partition(img_bgr, yolo_boxes, dilation_px=20):
    H, W     = img_bgr.shape[:2]
    combined = np.zeros((H, W), np.uint8)

    for (cx, cy, bw, bh) in yolo_boxes:
        x1 = max(0,   int((cx - bw/2) * W))
        y1 = max(0,   int((cy - bh/2) * H))
        x2 = min(W-1, int((cx + bw/2) * W))
        y2 = min(H-1, int((cy + bh/2) * H))
        if x2 - x1 < 4 or y2 - y1 < 4:
            combined[y1:y2, x1:x2] = 1
            continue
        gc = np.zeros((H, W), np.uint8)
        try:
            cv2.grabCut(img_bgr, gc, (x1, y1, x2-x1, y2-y1),
                        np.zeros((1, 65), np.float64),
                        np.zeros((1, 65), np.float64),
                        5, cv2.GC_INIT_WITH_RECT)
            fg = np.where((gc == 2) | (gc == 0), 0, 1).astype(np.uint8)
            if fg.sum() < (x2-x1)*(y2-y1)*0.05:
                fg[y1:y2, x1:x2] = 1
            combined = np.maximum(combined, fg)
        except Exception:
            combined[y1:y2, x1:x2] = 1

    if combined.max() > 0 and dilation_px > 0:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (dilation_px, dilation_px))
        combined = np.clip(cv2.dilate(combined, k, iterations=2), 0, 1).astype(np.uint8)
    if combined.max() == 0:
        combined = np.ones((H, W), np.uint8)

    return combined, 1 - combined

# ============================================================
# DATASET
# ============================================================
IMG_SIZE   = 128
MAX_IMAGES = 2000   # Use 2000 images for faster training; remove cap for full dataset

class VOCWatermarkDataset(Dataset):
    def __init__(self, img_dir, bbox_lookup, size=IMG_SIZE, max_imgs=MAX_IMAGES):
        self.img_dir     = img_dir
        self.bbox_lookup = bbox_lookup
        self.size        = size
        all_files        = sorted(glob.glob(os.path.join(img_dir, "*.jpg")))
        self.img_files   = [os.path.basename(f) for f in all_files[:max_imgs]]
        self.tf = T.Compose([T.Resize((size, size)), T.ToTensor()])

    def __len__(self): return len(self.img_files)

    def __getitem__(self, idx):
        fname = self.img_files[idx]
        S     = self.size
        pil   = Image.open(os.path.join(self.img_dir, fname)).convert("RGB")
        img_t = self.tf(pil)
        boxes = self.bbox_lookup.get(fname, [])
        bgr   = cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR)
        o, b  = grabcut_partition(bgr, boxes)
        om = cv2.resize(o.astype(np.float32), (S, S), interpolation=cv2.INTER_NEAREST)
        bm = cv2.resize(b.astype(np.float32), (S, S), interpolation=cv2.INTER_NEAREST)
        return (img_t,
                torch.from_numpy(om).unsqueeze(0),
                torch.from_numpy(bm).unsqueeze(0),
                fname)

full_ds = VOCWatermarkDataset(IMG_DIR, bbox_lookup)
n_val   = max(50, int(0.2 * len(full_ds)))
n_train = len(full_ds) - n_val
train_ds, val_ds = random_split(full_ds, [n_train, n_val],
                                generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True,
                          num_workers=2, pin_memory=(device == "cuda"))
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False,
                          num_workers=2, pin_memory=(device == "cuda"))

print(f"\nDataset   : {len(full_ds)} images (capped at {MAX_IMAGES})")
print(f"Train     : {n_train}   Val: {n_val}")
print(f"Batch size: 8   Train batches: {len(train_loader)}")

# ============================================================
# WATERMARK GENERATOR
# ============================================================
def generate_watermark(batch_size=1, seed=None):
    if seed is not None: torch.manual_seed(seed)
    wm = torch.randn(batch_size, 1, IMG_SIZE, IMG_SIZE)
    wm = F.avg_pool2d(wm, 9, 1, 4)
    wm = (wm - wm.min()) / (wm.max() - wm.min() + 1e-8)
    return (wm * 0.5).to(device)

W1 = generate_watermark(seed=42)
W2 = generate_watermark(seed=99)

# ============================================================
# STAGE 2 - DUAL-PATH AUTOENCODERS
# ============================================================
class LatentAutoencoder(nn.Module):
    def __init__(self, in_ch=3, base=32, latent_ch=64):
        super().__init__()
        self.enc1 = nn.Sequential(nn.Conv2d(in_ch, base, 3, 2, 1),
                                  nn.BatchNorm2d(base), nn.ReLU(True))
        self.enc2 = nn.Sequential(nn.Conv2d(base, base*2, 3, 2, 1),
                                  nn.BatchNorm2d(base*2), nn.ReLU(True))
        self.enc3 = nn.Sequential(nn.Conv2d(base*2, latent_ch, 3, 2, 1),
                                  nn.BatchNorm2d(latent_ch), nn.ReLU(True))
        self.wm_proj = nn.Conv2d(1, latent_ch, 1)
        self.u1    = nn.Parameter(torch.tensor(2.0))
        self.u2    = nn.Parameter(torch.tensor(0.5))
        self.gamma = nn.Parameter(torch.tensor(2.0))
        self.dec1  = nn.Sequential(
            nn.ConvTranspose2d(latent_ch, base*2, 3, 2, 1, 1),
            nn.BatchNorm2d(base*2), nn.ReLU(True))
        self.dec2  = nn.Sequential(
            nn.ConvTranspose2d(base*2, base, 3, 2, 1, 1),
            nn.BatchNorm2d(base), nn.ReLU(True))
        self.dec3  = nn.Sequential(
            nn.ConvTranspose2d(base, in_ch, 3, 2, 1, 1), nn.Sigmoid())

    def forward(self, img, wm, mode="object"):
        Z = self.enc3(self.enc2(self.enc1(img)))
        W = F.interpolate(self.wm_proj(wm), size=Z.shape[2:],
                          mode="bilinear", align_corners=False)
        Z_mod = Z + (self.u1*W + self.u2 if mode == "object" else self.gamma*W)
        return self.dec3(self.dec2(self.dec1(Z_mod)))

# ============================================================
# STAGE 3 - RECOMBINATION
# ============================================================
def recombine(wm_obj, wm_bg, om, bm):
    return torch.clamp(wm_obj*om + wm_bg*bm, 0, 1)

# ============================================================
# STAGE 4a - ATTACK LAYER (CNN + noise/jpeg/crop)
# ============================================================
class AttackLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.blur = nn.Conv2d(3, 3, 5, 1, 2, groups=3, bias=False)
        nn.init.dirac_(self.blur.weight)

    def forward(self, x, sigma=0.05, mode="noise"):
        if mode == "noise":
            return torch.clamp(self.blur(x + torch.randn_like(x)*sigma), 0, 1)
        elif mode == "jpeg":
            xq = torch.round(x*32)/32.0
            return torch.clamp(self.blur(xq), 0, 1)
        elif mode == "crop":
            B, C, H, W = x.shape
            m = int(0.1*H)
            return F.interpolate(x[:, :, m:H-m, m:W-m], size=(H, W),
                                 mode="bilinear", align_corners=False)
        return x

# ============================================================
# STAGE 4b - GAN GENERATOR (deeper, ch=128)
# ============================================================
class GANGenerator(nn.Module):
    def __init__(self, ch=128):
        super().__init__()
        self.head = nn.Sequential(nn.Conv2d(3, ch, 3, 1, 1), nn.ReLU(True))
        self.body = nn.Sequential(
            nn.Conv2d(ch, ch, 3, 1, 1), nn.BatchNorm2d(ch), nn.ReLU(True),
            nn.Conv2d(ch, ch, 3, 1, 1), nn.BatchNorm2d(ch), nn.ReLU(True),
            nn.Conv2d(ch, ch, 3, 1, 1), nn.BatchNorm2d(ch), nn.ReLU(True),
            nn.Conv2d(ch, ch, 3, 1, 1), nn.BatchNorm2d(ch), nn.ReLU(True),
            nn.Conv2d(ch, ch, 3, 1, 1), nn.BatchNorm2d(ch), nn.ReLU(True))
        self.tail = nn.Conv2d(ch, 3, 3, 1, 1)

    def forward(self, x):
        return torch.clamp(x + 0.15*self.tail(self.body(self.head(x))), 0, 1)

# ============================================================
# STAGE 5 - GAN DISCRIMINATOR (with Dropout to reduce overfit)
# ============================================================
class GANDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3,   32, 4, 2, 1), nn.LeakyReLU(0.2, True),
            nn.Dropout2d(0.1),
            nn.Conv2d(32,  64, 4, 2, 1), nn.BatchNorm2d(64),  nn.LeakyReLU(0.2, True),
            nn.Dropout2d(0.1),
            nn.Conv2d(64, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2, True),
            nn.Dropout2d(0.1),
            nn.Conv2d(128,  1, 4, 1, 1), nn.Sigmoid())

    def forward(self, x): return self.model(x)

# ============================================================
# INIT
# ============================================================
ae_obj = LatentAutoencoder().to(device)
ae_bg  = LatentAutoencoder().to(device)
attack = AttackLayer().to(device)
gen    = GANGenerator().to(device)
disc   = GANDiscriminator().to(device)

mse = nn.MSELoss()
bce = nn.BCELoss()
l1  = nn.L1Loss()

opt_ae_obj = torch.optim.Adam(ae_obj.parameters(), lr=1e-3, betas=(0.5, 0.999))
opt_ae_bg  = torch.optim.Adam(ae_bg.parameters(),  lr=1e-3, betas=(0.5, 0.999))
opt_gen    = torch.optim.Adam(gen.parameters(),     lr=1e-3, betas=(0.5, 0.999))
opt_disc   = torch.optim.Adam(disc.parameters(),    lr=5e-5, betas=(0.5, 0.999))

sched_obj = torch.optim.lr_scheduler.StepLR(opt_ae_obj, step_size=5, gamma=0.5)
sched_bg  = torch.optim.lr_scheduler.StepLR(opt_ae_bg,  step_size=5, gamma=0.5)
sched_gen = torch.optim.lr_scheduler.StepLR(opt_gen,    step_size=5, gamma=0.7)

# Label smoothing values (fixes generator collapse)
REAL_LABEL = 0.9   # was 1.0
FAKE_LABEL = 0.1   # was 0.0

EPOCHS = 25
SIGMA  = 0.05
history = {k: [] for k in ["ae_tr", "d_tr", "g_tr", "ae_val", "d_val", "g_val", "acc_tr", "acc_val"]}

def disc_acc(real_s, fake_s):
    return ((real_s > 0.5).float().mean().item() +
            (fake_s < 0.5).float().mean().item()) / 2.0

print("\n" + "="*80)
print(f"{'Ep':>3} | {'AE-tr':>7} {'D-tr':>7} {'G-tr':>7} | "
      f"{'AE-val':>7} {'D-val':>7} {'G-val':>7} | {'Acc-tr':>7} {'Acc-val':>7}")
print("-"*80)

# ============================================================
# TRAINING LOOP
# ============================================================
for epoch in range(EPOCHS):
    ae_obj.train(); ae_bg.train(); gen.train(); disc.train()
    t_ae = t_d = t_g = t_acc = 0.0
    steps = 0

    for img, om, bm, _ in train_loader:
        B   = img.size(0)
        img = img.to(device)
        om  = om.to(device)
        bm  = bm.to(device)
        w1  = W1.expand(B, -1, -1, -1)
        w2  = W2.expand(B, -1, -1, -1)

        # Stage 2 - embed
        wm_obj = ae_obj(img*om, w1, "object")
        wm_bg  = ae_bg(img*bm,  w2, "background")
        # Stage 3 - recombine
        I_w    = recombine(wm_obj, wm_bg, om, bm)

        # AE loss
        opt_ae_obj.zero_grad(); opt_ae_bg.zero_grad()
        loss_ae = (mse(wm_obj, img*om) + mse(wm_bg, img*bm)
                   - 0.05*(torch.mean(torch.abs(wm_obj - img*om))
                           + torch.mean(torch.abs(wm_bg - img*bm))))
        loss_ae.backward()
        opt_ae_obj.step(); opt_ae_bg.step()

        # Stage 4 - attack
        I_det = I_w.detach()
        att   = np.random.choice(["noise", "jpeg", "crop"])
        I_att = attack(I_det, SIGMA, att)
        I_fk  = gen(I_att)

        # Stage 5 - discriminator (label smoothing applied)
        opt_disc.zero_grad()
        sr = disc(I_det)
        sf = disc(I_fk.detach())
        loss_d = (bce(sr, torch.full_like(sr, REAL_LABEL)) +
                  bce(sf, torch.full_like(sf, FAKE_LABEL)))
        loss_d.backward(); opt_disc.step()

        # Generator
        opt_gen.zero_grad()
        fa = gen(I_att)
        loss_g = (bce(disc(fa), torch.full_like(sr, REAL_LABEL))
                  + l1(fa, I_det)*10)
        loss_g.backward(); opt_gen.step()

        t_ae  += loss_ae.item()
        t_d   += loss_d.item()
        t_g   += loss_g.item()
        t_acc += disc_acc(sr, sf)
        steps += 1

    sched_obj.step(); sched_bg.step(); sched_gen.step()

    # Validation
    ae_obj.eval(); ae_bg.eval(); gen.eval(); disc.eval()
    v_ae = v_d = v_g = v_acc = 0.0
    vsteps = 0
    with torch.no_grad():
        for img, om, bm, _ in val_loader:
            B   = img.size(0)
            img = img.to(device)
            om  = om.to(device)
            bm  = bm.to(device)
            w1  = W1.expand(B, -1, -1, -1)
            w2  = W2.expand(B, -1, -1, -1)
            wm_obj = ae_obj(img*om, w1, "object")
            wm_bg  = ae_bg(img*bm,  w2, "background")
            I_w    = recombine(wm_obj, wm_bg, om, bm)
            I_att  = attack(I_w, SIGMA, "noise")
            I_fk   = gen(I_att)
            sr     = disc(I_w)
            sf     = disc(I_fk)
            v_ae  += (mse(wm_obj, img*om) + mse(wm_bg, img*bm)).item()
            v_d   += (bce(sr, torch.full_like(sr, REAL_LABEL)) +
                      bce(sf, torch.full_like(sf, FAKE_LABEL))).item()
            v_g   += (bce(disc(gen(I_att)), torch.full_like(sr, REAL_LABEL))
                      + l1(gen(I_att), I_w)*10).item()
            v_acc += disc_acc(sr, sf)
            vsteps += 1

    def a(v, s): return v / s

    history["ae_tr"].append(a(t_ae, steps));   history["d_tr"].append(a(t_d, steps))
    history["g_tr"].append(a(t_g, steps));     history["acc_tr"].append(a(t_acc, steps))
    history["ae_val"].append(a(v_ae, vsteps)); history["d_val"].append(a(v_d, vsteps))
    history["g_val"].append(a(v_g, vsteps));   history["acc_val"].append(a(v_acc, vsteps))

    print(f"{epoch+1:>3} | "
          f"{history['ae_tr'][-1]:>7.4f} {history['d_tr'][-1]:>7.4f} {history['g_tr'][-1]:>7.4f} | "
          f"{history['ae_val'][-1]:>7.4f} {history['d_val'][-1]:>7.4f} {history['g_val'][-1]:>7.4f} | "
          f"{history['acc_tr'][-1]:>7.3f} {history['acc_val'][-1]:>7.3f}")

print("="*80)
print("TRAINING DONE")
print("="*80)

# ============================================================
# HELPER
# ============================================================
def to_np(t):
    return t.detach().cpu().squeeze().permute(1, 2, 0).numpy().clip(0, 1)

def save(fig, path, title=None):
    if title: fig.suptitle(title, fontsize=13, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.savefig(path, dpi=130, bbox_inches="tight")
    plt.show()
    print(f"Saved -> {path}")

ae_obj.eval(); ae_bg.eval(); gen.eval(); disc.eval()

with torch.no_grad():
    img_t, om_t, bm_t, fname_t = next(iter(val_loader))
    img_t = img_t[:1].to(device)
    om_t  = om_t[:1].to(device)
    bm_t  = bm_t[:1].to(device)

    # Run all stages
    wm_obj_t = ae_obj(img_t*om_t, W1, "object")
    wm_bg_t  = ae_bg(img_t*bm_t,  W2, "background")
    I_w_t    = recombine(wm_obj_t, wm_bg_t, om_t, bm_t)
    I_att_n  = attack(I_w_t, SIGMA, "noise")
    I_att_j  = attack(I_w_t, SIGMA, "jpeg")
    I_att_c  = attack(I_w_t, SIGMA, "crop")
    I_fk_t   = gen(I_att_n)

    REAL_SCORE = disc(I_w_t).mean().item()
    FAKE_SCORE = disc(I_fk_t).mean().item()

print(f"\nImage : {fname_t[0]}")
print(f"Real watermarked score : {REAL_SCORE:.4f}")
print(f"Fake generated score   : {FAKE_SCORE:.4f}")

# ============================================================
# OUTPUT 1 - STAGE 1: GrabCut Partitioning
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(to_np(img_t))
axes[0].set_title("Original image")
axes[0].axis("off")
axes[1].imshow(to_np(om_t.repeat(1, 3, 1, 1)))
axes[1].set_title("Object mask (GrabCut + dilated)")
axes[1].axis("off")
axes[2].imshow(to_np(bm_t.repeat(1, 3, 1, 1)))
axes[2].set_title("Background mask")
axes[2].axis("off")
save(fig, "/kaggle/working/output_stage1_grabcut.png",
     "STAGE 1 - Image Partitioning (GrabCut)")

# ============================================================
# OUTPUT 2 - STAGE 2: Autoencoder Watermark Embedding
# ============================================================
diff_obj = (torch.abs(wm_obj_t - img_t*om_t)*8).mean(1, keepdim=True)
diff_bg  = (torch.abs(wm_bg_t  - img_t*bm_t)*8).mean(1, keepdim=True)

fig, axes = plt.subplots(1, 6, figsize=(22, 4))
axes[0].imshow(to_np(img_t*om_t))
axes[0].set_title("Object region (input)")
axes[0].axis("off")
axes[1].imshow(to_np(wm_obj_t))
axes[1].set_title("WM Object I_o'\n(AE output)")
axes[1].axis("off")
axes[2].imshow(to_np(diff_obj.repeat(1, 3, 1, 1)))
axes[2].set_title("Embed diff obj x8\n(white=changed)")
axes[2].axis("off")
axes[3].imshow(to_np(img_t*bm_t))
axes[3].set_title("Background region (input)")
axes[3].axis("off")
axes[4].imshow(to_np(wm_bg_t))
axes[4].set_title("WM Background I_b'\n(AE output)")
axes[4].axis("off")
axes[5].imshow(to_np(diff_bg.repeat(1, 3, 1, 1)))
axes[5].set_title("Embed diff bg x8\n(white=changed)")
axes[5].axis("off")
save(fig, "/kaggle/working/output_stage2_autoencoder.png",
     "STAGE 2 - Dual-Path Autoencoder Watermark Embedding")

# ============================================================
# OUTPUT 3 - STAGE 3: Image Recombination
# ============================================================
diff_full = (torch.abs(I_w_t - img_t)*8).clamp(0, 1)
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(to_np(img_t))
axes[0].set_title("Original image")
axes[0].axis("off")
axes[1].imshow(to_np(I_w_t))
axes[1].set_title("Watermarked image I_w\n(object+bg merged)")
axes[1].axis("off")
axes[2].imshow(to_np(diff_full))
axes[2].set_title("Total embedding diff x8")
axes[2].axis("off")
save(fig, "/kaggle/working/output_stage3_recombination.png",
     "STAGE 3 - Image Recombination -> I_w")

# ============================================================
# OUTPUT 4 - STAGE 4: CNN Attack Layer
# ============================================================
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
axes[0].imshow(to_np(I_w_t))
axes[0].set_title("I_w (before attack)")
axes[0].axis("off")
axes[1].imshow(to_np(I_att_n))
axes[1].set_title("CNN: Gaussian noise\n(sigma=0.05)")
axes[1].axis("off")
axes[2].imshow(to_np(I_att_j))
axes[2].set_title("CNN: JPEG simulation\n(quantise 32 levels)")
axes[2].axis("off")
axes[3].imshow(to_np(I_att_c))
axes[3].set_title("CNN: Crop-resize\n(10% margin crop)")
axes[3].axis("off")
save(fig, "/kaggle/working/output_stage4_attack.png",
     "STAGE 4a - CNN Attack Simulation (3 attack types)")

# ============================================================
# OUTPUT 5 - STAGE 4b: GAN Generator
# ============================================================
I_fk_j = gen(I_att_j)
I_fk_c = gen(I_att_c)
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
axes[0].imshow(to_np(I_att_n))
axes[0].set_title("Attacked input (noise)")
axes[0].axis("off")
axes[1].imshow(to_np(I_fk_t))
axes[1].set_title("GAN Generator output\n(from noise attack)")
axes[1].axis("off")
axes[2].imshow(to_np(I_fk_j))
axes[2].set_title("GAN Generator output\n(from JPEG attack)")
axes[2].axis("off")
axes[3].imshow(to_np(I_fk_c))
axes[3].set_title("GAN Generator output\n(from crop attack)")
axes[3].axis("off")
save(fig, "/kaggle/working/output_stage4b_generator.png",
     "STAGE 4b - GAN Generator Output (refinement network)")

# ============================================================
# OUTPUT 6 - STAGE 5: GAN DISCRIMINATOR - FINAL VERDICT
# ============================================================
VERDICT_REAL = "WATERMARKED REAL ✓" if REAL_SCORE >= 0.5 else "WATERMARKED FAKE ✗"
VERDICT_FAKE = "WATERMARKED FAKE ✗" if FAKE_SCORE < 0.5  else "WATERMARKED REAL ✓"
COLOR_R      = "green" if REAL_SCORE >= 0.5 else "red"
COLOR_F      = "red"   if FAKE_SCORE < 0.5  else "green"

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
fig.suptitle("STAGE 5 - GAN DISCRIMINATOR: FINAL VERDICT", fontsize=14, fontweight="bold")

axes[0].imshow(to_np(I_w_t))
axes[0].set_title(f"Input: Real watermarked image\nScore: {REAL_SCORE:.4f}", fontsize=10)
axes[0].set_xlabel(VERDICT_REAL, fontsize=13, color=COLOR_R, fontweight="bold")
axes[0].axis("off")

axes[1].imshow(to_np(I_fk_t))
axes[1].set_title(f"Input: GAN-generated fake image\nScore: {FAKE_SCORE:.4f}", fontsize=10)
axes[1].set_xlabel(VERDICT_FAKE, fontsize=13, color=COLOR_F, fontweight="bold")
axes[1].axis("off")

# Score bar visual
for ax, score, label in [(axes[0], REAL_SCORE, "Real"), (axes[1], FAKE_SCORE, "Fake")]:
    axins = ax.inset_axes([0, -0.18, 1, 0.08])
    axins.barh([0], [score],   color="#3B6D11", height=0.5)
    axins.barh([0], [1-score], color="#e0e0e0", height=0.5, left=score)
    axins.set_xlim(0, 1)
    axins.axvline(0.5, color="gray", lw=1, ls="--")
    axins.axis("off")

plt.tight_layout()
plt.savefig("/kaggle/working/output_stage5_FINAL_VERDICT.png", dpi=130, bbox_inches="tight")
plt.show()
print("Saved -> /kaggle/working/output_stage5_FINAL_VERDICT.png")

# ============================================================
# OUTPUT 7 - LOSS CURVES + ACCURACY
# ============================================================
ep = range(1, EPOCHS+1)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(ep, history["ae_tr"],  "b-o",  ms=3, label="AE train")
axes[0].plot(ep, history["ae_val"], "b--s", ms=3, label="AE val")
axes[0].set_title("Autoencoder Loss"); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(ep, history["d_tr"],  "r-o",  ms=3, label="Disc train")
axes[1].plot(ep, history["d_val"], "r--s", ms=3, label="Disc val")
axes[1].plot(ep, history["g_tr"],  "g-o",  ms=3, label="Gen train")
axes[1].plot(ep, history["g_val"], "g--s", ms=3, label="Gen val")
axes[1].set_title("Discriminator & Generator Loss"); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(ep, history["acc_tr"],  "b-o",  ms=3, label="Train accuracy")
axes[2].plot(ep, history["acc_val"], "r-s",  ms=3, label="Val accuracy")
axes[2].axhline(0.8, color="gray", ls="--", alpha=0.5, label="80% target")
axes[2].set_ylim(0, 1.05)
axes[2].set_title("Discriminator Accuracy")
axes[2].legend(); axes[2].grid(alpha=0.3)

for ax in axes: ax.set_xlabel("Epoch")
save(fig, "/kaggle/working/output_loss_curves.png",
     "Training vs Validation - Loss Curves & Accuracy")

# ============================================================
# ROBUSTNESS TEST
# ============================================================
print("\n-- Robustness Testing --")
with torch.no_grad():
    for mode in ["noise", "jpeg", "crop"]:
        I_a  = attack(I_w_t, SIGMA, mode)
        I_f  = gen(I_a)
        sr   = disc(I_w_t).mean().item()
        sf   = disc(I_f).mean().item()
        acc  = disc_acc(disc(I_w_t), disc(I_f))
        verd = "REAL✓" if sr >= 0.5 else "FAKE✗"
        print(f"  [{mode:5s}]  Real:{sr:.3f}({verd})  Fake:{sf:.3f}  Acc:{acc:.3f}")

# ============================================================
# INFERENCE SPEED
# ============================================================
print("\n-- Inference Speed --")
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device)
with torch.no_grad():
    for _ in range(5):
        _ = disc(ae_obj(dummy, W1, "object"))
t0 = time.perf_counter()
with torch.no_grad():
    for _ in range(50):
        wo = ae_obj(dummy, W1, "object")
        wb = ae_bg(dummy,  W2, "background")
        Iw = recombine(wo, wb,
                       torch.ones(1, 1, IMG_SIZE, IMG_SIZE).to(device),
                       torch.zeros(1, 1, IMG_SIZE, IMG_SIZE).to(device))
        _  = disc(Iw)
ms = (time.perf_counter() - t0) / 50 * 1000
print(f"  {ms:.2f} ms / image   ({1000/ms:.0f} img/sec)")

# ============================================================
# FINAL VERDICT PRINT
# ============================================================
gap = history["acc_tr"][-1] - history["acc_val"][-1]
print("\n" + "="*60)
print("  FINAL EVALUATION SUMMARY")
print("="*60)
rows = [
    ("Real WM score",          f"{REAL_SCORE:.4f}",              "-> 1.0",  REAL_SCORE >= 0.7),
    ("Fake score",             f"{FAKE_SCORE:.4f}",              "-> 0.0",  FAKE_SCORE <= 0.3),
    ("Discriminator accuracy", f"{history['acc_val'][-1]:.4f}",  "> 0.80",  history['acc_val'][-1] >= 0.8),
    ("Gen loss (val)",         f"{history['g_val'][-1]:.4f}",    "< 2.0",   history['g_val'][-1] < 2.0),
    ("Train/val acc gap",      f"{gap:.4f}",                     "< 0.10",  abs(gap) < 0.10),
    ("Inference speed",        f"{ms:.2f} ms",                   "< 50ms",  ms < 50),
]
for name, val, target, passed in rows:
    icon = "OK" if passed else "!!"
    print(f"  [{icon}]  {name:<35} {val:>10}  (target {target})")
print("="*60)
print(f"\n  REAL watermarked image  -> score {REAL_SCORE:.4f} -> {VERDICT_REAL}")
print(f"  GAN fake image          -> score {FAKE_SCORE:.4f} -> {VERDICT_FAKE}")
print("="*60)

# Save checkpoint
torch.save({
    "ae_obj":  ae_obj.state_dict(),
    "ae_bg":   ae_bg.state_dict(),
    "gen":     gen.state_dict(),
    "disc":    disc.state_dict(),
    "W1":      W1.cpu(),
    "W2":      W2.cpu(),
    "history": history
}, "/kaggle/working/checkpoint_v3.pth")

print("\nOutput files -> /kaggle/working/")
print("   output_stage1_grabcut.png")
print("   output_stage2_autoencoder.png")
print("   output_stage3_recombination.png")
print("   output_stage4_attack.png")
print("   output_stage4b_generator.png")
print("   output_stage5_FINAL_VERDICT.png   <- most important")
print("   output_loss_curves.png")
print("   checkpoint_v3.pth")

Device : cuda
GPU  : Tesla T4
VRAM : 15.6 GB

Images     : /kaggle/input/datasets/huanghanchina/pascal-voc-2012/VOC2012/JPEGImages  (17125 files)
Annotations: /kaggle/input/datasets/huanghanchina/pascal-voc-2012/VOC2012/Annotations  (17125 files)

Parsing annotations...
Parsed 17125 images with bounding boxes

Dataset   : 2000 images (capped at 2000)
Train     : 1600   Val: 400
Batch size: 8   Train batches: 200

 Ep |   AE-tr    D-tr    G-tr |  AE-val   D-val   G-val |  Acc-tr Acc-val
--------------------------------------------------------------------------------
  1 |  0.0440  1.3901  1.6424 |  0.0140  1.3689  1.4812 |   0.525   0.593
  2 |  0.0075  1.3883  1.5429 |  0.0103  1.3941  1.6127 |   0.525   0.467
  3 |  0.0052  1.3850  1.5537 |  0.0068  1.4092  1.8583 |   0.532   0.414
  4 |  0.0046  1.3736  1.5684 |  0.0062  1.3994  2.0627 |   0.558   0.450
  5 |  0.0040  1.3650  1.5800 |  0.0054  1.3893  2.1370 |   0.579   0.489
  6 |  0.0032  1.3579  1.5938 |  0.0051  1.3567  1.7899 | 